In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import lit

spark.conf.set(
  "fs.azure.account.key.adlsvenproject1.dfs.core.windows.net",
  dbutils.secrets.get(scope="secret-scope-azure-live-project-1", key="storage-key")
)

bronze_product_path = "abfss://datalake-ven-project1@adlsvenproject1.dfs.core.windows.net/bronze/product/"

# List folders
folders = dbutils.fs.ls(bronze_product_path)

# Extract load_date values
dates = [f.name.replace("load_date=", "").replace("/", "") 
         for f in folders if "load_date=" in f.name]

# Get latest
latest_date = sorted(dates, reverse=True)[0]

print("Latest load_date:", latest_date)

# Read ONLY latest partition
df_product = spark.read.parquet(f"{bronze_product_path}load_date={latest_date}/")
df_product = df_product.withColumn("load_date", lit(latest_date))

display(df_product)

In [0]:
bronze_detail_path = "abfss://datalake-ven-project1@adlsvenproject1.dfs.core.windows.net/bronze/sales_order_detail/"

folders = dbutils.fs.ls(bronze_detail_path)

dates = sorted([
    f.name.replace("load_date=", "").replace("/", "")
    for f in folders if "load_date=" in f.name
])

print(dates)

In [0]:
bronze_detail_path = "abfss://datalake-ven-project1@adlsvenproject1.dfs.core.windows.net/bronze/sales_order_detail/"

# Read ONLY available partitions
folders = dbutils.fs.ls(bronze_detail_path)

dates = sorted([
    f.name.replace("load_date=", "").replace("/", "")
    for f in folders if "load_date=" in f.name
])

# For now → pick all available (later we optimize with watermark)
paths = [f"{bronze_detail_path}load_date={d}/" for d in dates]

df_detail = spark.read.parquet(*paths)

display(df_detail)

In [0]:
from pyspark.sql.functions import input_file_name, regexp_extract

df_detail = df_detail.withColumn(
    "load_date",
    regexp_extract(input_file_name(), r'load_date=(\d{4}-\d{2}-\d{2})', 1)
)

display(df_detail)

In [0]:
latest_date = max(dates)

df_detail_latest = df_detail.filter(df_detail.load_date == latest_date)

display(df_detail_latest)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window = Window.partitionBy("SalesOrderDetailID") \
               .orderBy(col("ModifiedDate").desc())

df_detail_clean = df_detail_latest.withColumn(
    "rn",
    row_number().over(window)
).filter(col("rn") == 1).drop("rn")

display(df_detail_clean)

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import lit
from pyspark.sql.functions import input_file_name, regexp_extract

spark.conf.set(
  "fs.azure.account.key.adlsvenproject1.dfs.core.windows.net",
  dbutils.secrets.get(scope="secret-scope-azure-live-project-1", key="storage-key")
)

bronze_header_path = "abfss://datalake-ven-project1@adlsvenproject1.dfs.core.windows.net/bronze/sales_order_header/"

folders = dbutils.fs.ls(bronze_header_path)

dates = sorted([
    f.name.replace("load_date=", "").replace("/", "")
    for f in folders if "load_date=" in f.name
])

latest_date = dates[-1]

df_header = spark.read.parquet(
    f"{bronze_header_path}load_date={latest_date}/"
)

#display(df_header)


df_header = df_header.withColumn(
    "load_date",
    regexp_extract(input_file_name(), r'load_date=(\d{4}-\d{2}-\d{2})', 1)
)

display(df_header)

In [0]:
df_final = df_detail_clean.select(
    "SalesOrderDetailID",
    "SalesOrderID",
    "ProductID",
    "OrderQty",
    "LineTotal",
    "ModifiedDate"
).join(
    df_header.select(
        "SalesOrderID",
        "OrderDate",
        "CustomerID"
    ),
    on="SalesOrderID",
    how="left"
).join(
    df_product.select(
        "ProductID",
        "Name"   # adjust if your column name differs
    ),
    on="ProductID",
    how="left"
)

display(df_final)

In [0]:
from delta.tables import DeltaTable

silver_path = "abfss://datalake-ven-project1@adlsvenproject1.dfs.core.windows.net/silver/sales_data/"

# Create table if not exists (first run)
if not DeltaTable.isDeltaTable(spark, silver_path):
    df_final.write.format("delta").mode("overwrite").save(silver_path)

# Load Delta table
delta_table = DeltaTable.forPath(spark, silver_path)
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
# Merge
merge_result = delta_table.alias("t").merge(
    df_final.alias("s"),
    "t.SalesOrderDetailID = s.SalesOrderDetailID"
).whenMatchedUpdate(
    condition="""
        t.ModifiedDate IS NULL OR
        s.ModifiedDate > t.ModifiedDate
    """,
    set={c: f"s.{c}" for c in df_final.columns}
).whenNotMatchedInsertAll() \
.execute()

display(merge_result)

In [0]:
#df_final.write.format("delta") \
#    .mode("overwrite") \
#    .option("overwriteSchema", "true") \
#    .save(silver_path)

In [0]:
df_silver = spark.read.format("delta").load(silver_path)

print("Silver count:", df_silver.count())